In [7]:
import sys
import os

# 1. Direct Python to the root project folder
sys.path.append(os.path.abspath(os.path.join('..')))

In [9]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [6]:
from app.ingest import load_faq_data, build_index
documents = load_faq_data()

index = build_index(documents)

Ecommerce FAQ With Ids Generated Successfully!!


In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [7]:
q = ground_truth[10]
q

{'question': 'How do I find the tracking info for my order after I log in?',
 'document': 'LOdDd5N3'}

In [8]:
doc_idx[q['document']]

{'question': 'How can I track my order?',
 'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.",
 'id': 'LOdDd5N3'}

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from app.evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [11]:
q['question']

'How do I find the tracking info for my order after I log in?'

In [12]:
answer = assistant.rag(q['question'])

In [13]:
print(answer)

Log into your account and go to the **Order History** section. That’s where you’ll find the tracking information for your shipment.


In [14]:
doc_idx[q['document']]

{'question': 'How can I track my order?',
 'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.",
 'id': 'LOdDd5N3'}

In [15]:
assistant.total_cost()

0.00038625

In [16]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

"You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment."

In [17]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I find the tracking info for my order after I log in?',
 'answer_llm': 'Log into your account and go to the **Order History** section. That’s where you’ll find the tracking information for your shipment.',
 'answer_orig': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.",
 'document': 'LOdDd5N3'}

In [18]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [19]:
answer_record = generate_rag_answer(q)
answer_record

{'question': 'How do I find the tracking info for my order after I log in?',
 'answer_llm': 'After logging in, go to your **Order History** section. That’s where you’ll find the **tracking information** for your shipment.',
 'answer_orig': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.",
 'document': 'LOdDd5N3'}

In [20]:
assistant.reset_usage()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from app.evaluation_utils import map_progress

In [22]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth[:100], generate_rag_answer)

  0%|          | 0/100 [00:00<?, ?it/s]

In [23]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [24]:
assistant.total_cost()

0.035456999999999995

In [25]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("../data/rag-answers.csv", index=False)

In [26]:
df_answers

,question,answer_llm,answer_orig,document
0,How do I make an account on your website?,"To create an account, click on the **“Sign Up”...","To create an account, click on the 'Sign Up' b...",qDNpG85o
1,Where do I click to sign up for a new account?,Click the **“Sign Up”** button in the **top ri...,"To create an account, click on the 'Sign Up' b...",qDNpG85o
2,What’s the easiest way to register on your site?,I don't know.,"To create an account, click on the 'Sign Up' b...",qDNpG85o
3,Can you tell me how to create an account online?,"To create an account, click the **“Sign Up”** ...","To create an account, click on the 'Sign Up' b...",qDNpG85o
4,How do I complete the account registration pro...,"To complete the account registration process, ...","To create an account, click on the 'Sign Up' b...",qDNpG85o
...,...,...,...,...
95,Where do I go on the site if I want to write a...,Go to the product page on the website and clic...,"To leave a product review, navigate to the pro...",4riZFS2i
96,How do I submit feedback and a rating for a pr...,"To submit feedback and a rating for a product,...","To leave a product review, navigate to the pro...",4riZFS2i
97,Is there a button I need to click to leave a r...,"Yes. To leave a review, go to the product page...","To leave a product review, navigate to the pro...",4riZFS2i
98,"Can I review a product directly from its page,...","Yes. To leave a review, go to the product page...","To leave a product review, navigate to the pro...",4riZFS2i


In [27]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )


In [28]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a customer
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [29]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from app.evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [32]:
import pandas as pd

df_answers = pd.read_csv("../data/rag-answers.csv")
answers = df_answers.to_dict(orient="records")

In [33]:
rec = answers[0]
rec

{'question': 'How do I make an account on your website?',
 'answer_llm': 'To create an account, click on the **“Sign Up”** button in the **top right corner** of the website and follow the instructions to complete the registration process.',
 'answer_orig': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.",
 'document': 'qDNpG85o'}

In [34]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

print(prompt)

Question:
How do I make an account on your website?

Original Answer (ground truth):
To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.

AI Answer:
To create an account, click on the **“Sign Up”** button in the **top right corner** of the website and follow the instructions to complete the registration process.


In [ ]:
from app.evaluation_utils import llm_structured_retry

eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning="The AI answer conveys the same instructions as the ground truth: click the 'Sign Up' button in the top right corner and follow the registration steps. It is semantically equivalent.", score='good')

In [36]:
print(eval_result)

reasoning="The AI answer conveys the same instructions as the ground truth: click the 'Sign Up' button in the top right corner and follow the registration steps. It is semantically equivalent." score='good'


In [37]:
calc_price(usage)

{'input_cost': 0.00022725,
 'output_cost': 0.000234,
 'total_cost': 0.00046124999999999996}

In [38]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [39]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning="The AI answer matches the ground truth exactly in meaning: it instructs the user to click the 'Sign Up' button in the top right corner and follow the registration instructions. No key information is missing or incorrect.", score='good')

In [40]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [41]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/100 [00:00<?, ?it/s]

In [42]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [43]:
df_eval = pd.DataFrame(evaluations)

In [44]:
calc_total_price(usages)

0.05071575

In [45]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 88/100 = 88.00%


In [46]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
2,What’s the easiest way to register on your site?,qDNpG85o,bad,The AI answer fails to provide the registratio...
7,Can I pay for my order with PayPal?,2mJCxUVM,bad,The ground truth says PayPal is accepted as a ...
8,Which card types are accepted for checkout?,2mJCxUVM,bad,The AI answer does not convey the ground truth...
13,What do I need to do to check the status of my...,LOdDd5N3,bad,The AI answer does not provide the required in...
19,Do you accept returns after 30 days from the d...,vxC9JJ39,bad,The AI answer does not answer the question or ...


In [48]:
df_eval.to_csv("../data/rag-answers-evaluations.csv", index=False)